# Build `product_characteristics.dta` from lenth9-format transactions

Input columns required: `firm_id`, `product_id`, `is_output`, `year`, `v`

Output: 2,778 rows × 11 cols, sorted by `num_firms` desc.

In [ ]:
import pandas as pd

INPUT  = "lenth9.dta"                   # 改成 VM 上的实际路径
OUTPUT = "product_characteristics.dta"

# ─── 1. 读取交易数据，清洗 ──────────────────────────────
df = pd.read_stata(INPUT)
df = df.dropna(subset=["year", "firm_id", "product_id", "is_output", "v"])
df = df[df["v"] >= 0]

# ─── 2. 分别 sum output / input 到 firm-product-year 级 ──────────────
out = (df[df["is_output"] == 1]
       .groupby(["year", "firm_id", "product_id"])["v"].sum()
       .reset_index()
       .rename(columns={"v": "total_output"}))

inp = (df[df["is_output"] == 0]
       .groupby(["year", "firm_id", "product_id"])["v"].sum()
       .reset_index()
       .rename(columns={"v": "total_input"}))

# 以 output 端为主，input 缺失填 0
fpy = out.merge(inp, on=["year", "firm_id", "product_id"], how="left")
fpy["total_input"] = fpy["total_input"].fillna(0)

# outsourcing = min(input, output); production = output - outsourcing
fpy["outsourcing_value"] = fpy[["total_input", "total_output"]].min(axis=1)
fpy["production_value"]  = fpy["total_output"] - fpy["outsourcing_value"]

# ─── 3. 产品层面汇总（跨 firm-year）───────────────────────
chars = fpy.groupby("product_id", as_index=False).agg(
    total_output      = ("total_output",      "sum"),
    total_outsourcing = ("outsourcing_value", "sum"),
    total_production  = ("production_value",  "sum"),
    num_years         = ("year",              "nunique"),
)

# ─── 4. 跨年去重的 firm 计数 ─────────────────────────────
num_firms = (fpy[["product_id", "firm_id"]].drop_duplicates()
             .groupby("product_id").size().rename("num_firms"))

num_firms_os = (fpy[fpy["outsourcing_value"] > 0][["product_id", "firm_id"]]
                .drop_duplicates()
                .groupby("product_id").size().rename("num_firms_outsourcing"))

chars = chars.merge(num_firms,    on="product_id", how="left")
chars = chars.merge(num_firms_os, on="product_id", how="left")
chars["num_firms_outsourcing"] = chars["num_firms_outsourcing"].fillna(0).astype(int)

# ─── 5. 派生比率 ───────────────────────────────────────
chars["outsourcing_intensity"]  = (chars["total_outsourcing"] / chars["total_output"]).fillna(0)
chars["avg_output_per_firm"]    = chars["total_output"] / chars["num_firms"]
chars["avg_output_per_year"]    = chars["total_output"] / chars["num_years"]
chars["pct_firms_outsourcing"]  = (chars["num_firms_outsourcing"] / chars["num_firms"]).fillna(0)

# ─── 6. 排序 + 保存 ────────────────────────────────────
chars = chars.sort_values("num_firms", ascending=False).reset_index(drop=True)
chars.to_stata(OUTPUT, write_index=False)

print(f"✓ {len(chars):,} products → {OUTPUT}")